Importamos los módulos necesarios para la ejecución del notebook

In [6]:
import random
import time
import cv2
import numpy as np
import mediapipe as mp
from mediapipe.tasks import python as mp_python
from mediapipe.tasks.python import vision
from PIL import Image, ImageDraw, ImageFont

Configuración de parámetros

In [7]:
FACE_MODEL = "./models/face_landmarker.task"
MAX_FACES = 3

PROP_PATHS = [
    ("./assets/monkeys/monkey_happy.jpg", 1.0),
    ("./assets/monkeys/monkey_mad.jpg", 1.0),
    ("./assets/monkeys/monkey_sad.jpg", 1.0),
    ("./assets/monkeys/monkey_sleepy.jpg", 1.0),
    ("./assets/monkeys/monkey_stare.jpg", 1.0),
    ("./assets/monkeys/monkey_surprised.jpg", 1.0),
    ("./assets/monkeys/monkey_thinking.jpg", 1.0),
    ("./assets/monkeys/monkey_uwu.jpg", 1.0),
]
MONKEY_WEIGHT = 1.0
MONKEY_VIDEO = None
TITLE_TEXT = "¿Qué mono eres?"

CARD_W = 2.4
LIFT = 2.3
CARD_PX = 192

SPIN_STEPS = 16
SPIN_FIRST, SPIN_LAST = 0.05, 0.5
AUTO_START = True

# --- Geometría y estabilidad ---
NOSE_DEPTH = 0.5
FEATHER = 0.8
HOLD = 3
MIN_CUTOFF = 1.0
BETA = 0.02

Funciones auxiliares de carga de imágenes, dibujado y geometría

In [8]:
KEYS = ["left_eye", "right_eye", "nose", "mouth_left", "mouth_right"]


def load_rgba(path):
    im = cv2.imread(path, cv2.IMREAD_UNCHANGED)
    assert im is not None, f"No se encuentra {path}"
    if im.ndim == 2:
        im = cv2.cvtColor(im, cv2.COLOR_GRAY2BGR)
    if im.shape[2] == 3:
        a = (~np.all(im > 240, axis=2)).astype(np.uint8) * 255
        im = np.dstack([im, a])
    return im


def prepare(im):
    """BGRA uint8 -> float32 con RGB premultiplicado por alfa (evita bordes oscuros) y borde suave."""
    f = im.astype(np.float32)
    f[:, :, :3] *= f[:, :, 3:4] / 255.0
    if FEATHER > 0:
        f = cv2.GaussianBlur(f, (0, 0), FEATHER)
    return f


def overlay(frame, sprite, src_pt, dst_pt, sx, sy, angle_deg, gain=1.0):
    """Pinta el sprite sobre frame: src_pt cae en dst_pt, con escala (sx, sy) y rotación angle_deg
    (horaria en pantalla). Solo procesa la zona afectada."""
    a = np.radians(angle_deg)
    A = np.array([[np.cos(a), -np.sin(a)], [np.sin(a), np.cos(a)]]) @ np.diag([sx, sy])
    t = np.asarray(dst_pt, float) - A @ np.asarray(src_pt, float)
    h, w = sprite.shape[:2]
    corners = np.array([[0, 0], [w, 0], [w, h], [0, h]], float) @ A.T + t
    x0, y0 = np.floor(corners.min(0)).astype(int)
    x1, y1 = np.ceil(corners.max(0)).astype(int)
    H, W = frame.shape[:2]
    x0, y0, x1, y1 = max(x0, 0), max(y0, 0), min(x1, W), min(y1, H)
    if x1 <= x0 or y1 <= y0:
        return
    M = np.hstack([A, (t - [x0, y0])[:, None]])
    warped = cv2.warpAffine(sprite, M, (x1 - x0, y1 - y0), flags=cv2.INTER_LINEAR,
                            borderMode=cv2.BORDER_CONSTANT, borderValue=(0, 0, 0, 0))
    alpha = warped[:, :, 3:4] / 255.0
    roi = frame[y0:y1, x0:x1]
    roi[:] = np.clip(warped[:, :, :3] * gain + roi * (1 - alpha), 0, 255).astype(np.uint8)


def face_pose(kp):
    """Punto medio de los ojos, vector 'arriba', ángulo, escala corregida por giro y cos(giro)."""
    le, re = kp["left_eye"], kp["right_eye"]
    if le[0] > re[0]:
        le, re = re, le
    d = re - le
    iod = max(float(np.hypot(*d)), 1e-3)
    u = d / iod
    angle = np.degrees(np.arctan2(d[1], d[0]))
    mid = (le + re) / 2
    up = np.array([u[1], -u[0]])
    
    t = np.dot(kp["nose"] - mid, u) / iod
    cy = float(np.clip(1.0 / np.sqrt(1.0 + (t / NOSE_DEPTH) ** 2), 0.6, 1.0))
    return mid, up, angle, iod / cy, cy

class OneEuro:
    """Suaviza sin retraso: poco cutoff con la cabeza quieta, más cutoff cuanto más rápido se mueve."""
    def __init__(self, min_cutoff=MIN_CUTOFF, beta=BETA, d_cutoff=1.0):
        self.min_cutoff, self.beta, self.d_cutoff = min_cutoff, beta, d_cutoff
        self.x = self.dx = self.t = None

    @staticmethod
    def _alpha(cutoff, dt):
        return 1.0 / (1.0 + 1.0 / (2 * np.pi * cutoff * dt))

    def __call__(self, x, t):
        if self.x is None:
            self.x, self.dx, self.t = x.copy(), np.zeros_like(x), t
            return self.x
        dt = max(t - self.t, 1e-3)
        a = self._alpha(self.d_cutoff, dt)
        self.dx = a * (x - self.x) / dt + (1 - a) * self.dx
        cutoff = self.min_cutoff + self.beta * np.linalg.norm(self.dx, axis=-1, keepdims=True)
        a = self._alpha(cutoff, dt)
        self.x = a * x + (1 - a) * self.x
        self.t = t
        return self.x


class Track:
    """Una cara seguida: landmarks suavizados, extrapolación entre detecciones y mantenimiento si falla."""
    def __init__(self):
        self.f, self.miss, self.t = OneEuro(), 0, 0.0
        self.slot = None

    def update(self, kp, t):
        self.f(np.array([kp[k] for k in KEYS], float), t)
        self.miss, self.t = 0, t

    def kp(self, t):
        x = self.f.x
        if self.miss == 0:
            x = x + self.f.dx * min(max(t - self.t, 0.0), 0.12)
        return dict(zip(KEYS, x))

    def center(self):
        return (self.f.x[0] + self.f.x[1]) / 2

    def iod(self):
        return np.linalg.norm(self.f.x[0] - self.f.x[1])


def update_tracks(tracks, faces, t):
    """Asocia cada cara detectada con el seguimiento más cercano; crea o descarta seguimientos."""
    used = set()
    for kp in faces:
        mid = (kp["left_eye"] + kp["right_eye"]) / 2
        best, bd = None, 1e9
        for i, tr in enumerate(tracks):
            d = np.linalg.norm(tr.center() - mid)
            if i not in used and d < bd and d < 2 * tr.iod():
                best, bd = i, d
        if best is None:
            tracks.append(Track())
            best = len(tracks) - 1
        tracks[best].update(kp, t)
        used.add(best)
    for i, tr in enumerate(tracks):
        if i not in used:
            tr.miss += 1
    tracks[:] = [tr for tr in tracks if tr.miss <= HOLD]

Props

In [9]:
PROPS = []
for _path, _w in PROP_PATHS:
    _im = load_rgba(_path)
    PROPS.append((prepare(_im), _im.shape[1]))
WEIGHTS = [w for _, w in PROP_PATHS] + [MONKEY_WEIGHT]
MONKEY = len(PROPS)

_mcap = cv2.VideoCapture(MONKEY_VIDEO) if MONKEY_VIDEO else None

def make_title(text, px=64):
    """Texto blanco con contorno negro como sprite premultiplicado (PIL admite tildes y ¿)."""
    if not text:
        return None
    font = None
    for name in ("arialbd.ttf", "Arial Bold.ttf", "Arial.ttf", "arial.ttf", "DejaVuSans-Bold.ttf", "DejaVuSans.ttf",
                 "/System/Library/Fonts/Supplemental/Arial Bold.ttf"):
        try:
            font = ImageFont.truetype(name, px)
            break
        except OSError:
            pass
    if font is None:
        try:
            font = ImageFont.load_default(size=px)
        except TypeError:
            font = ImageFont.load_default()
    pad = 14
    l, t, r, b = font.getbbox(text)
    img = Image.new("RGBA", (r - l + 2 * pad, b - t + 2 * pad), (0, 0, 0, 0))
    ImageDraw.Draw(img).text((pad - l, pad - t), text, font=font, fill=(255, 255, 255, 255),
                             stroke_width=6, stroke_fill=(0, 0, 0, 255))
    return prepare(np.ascontiguousarray(np.array(img)[:, :, [2, 1, 0, 3]]))


TITLE = make_title(TITLE_TEXT)


def start_spin(tr, now):
    """Elige el resultado final (con pesos). La secuencia que gira solo contiene props:
    el mono nunca pasa por la ruleta, aparece directamente al parar."""
    n_props = len(PROPS)
    target = random.choices(range(n_props + 1), weights=WEIGHTS)[0]
    seq = []
    for i in range(SPIN_STEPS - 1):
        banned = {seq[-1]} if seq else set()
        if i == SPIN_STEPS - 2 and target != MONKEY:
            banned.add(target)
        seq.append(random.choice([j for j in range(n_props) if j not in banned] or list(range(n_props))))
    if target == MONKEY:
        seq.append(random.choice([j for j in range(n_props) if j != seq[-1]] or [0]))
    else:
        seq.append(target)
    times = np.cumsum(np.geomspace(SPIN_FIRST, SPIN_LAST, SPIN_STEPS))
    tr.slot = dict(t0=now, seq=seq, times=times, target=target)

def face_crop(raw, kp):
    """Recorta la cara (enderezada según los ojos) del frame SIN filtros."""
    mid, _, angle, iod, _ = face_pose(kp)
    M = cv2.getRotationMatrix2D((float(mid[0]), float(mid[1])), angle, CARD_PX / (3.2 * iod))
    M[:, 2] += [CARD_PX / 2 - mid[0], CARD_PX * 0.42 - mid[1]]
    return cv2.warpAffine(raw, M, (CARD_PX, CARD_PX), borderMode=cv2.BORDER_REPLICATE)


def monkey_img(raw, kp):
    """Imagen de la carta mono: clip de vídeo en bucle, o la cara en directo."""
    if _mcap is not None:
        ok, fr = _mcap.read()
        if not ok:
            _mcap.set(cv2.CAP_PROP_POS_FRAMES, 0)
            ok, fr = _mcap.read()
        if ok:
            h, w = fr.shape[:2]
            s = min(h, w)
            return fr[(h - s) // 2:(h + s) // 2, (w - s) // 2:(w + s) // 2]
    return face_crop(raw, kp)


def make_card(img):
    """Imagen cuadrada con borde blanco, lista para overlay()."""
    px = CARD_PX
    card = np.dstack([cv2.resize(img, (px, px)), np.full((px, px), 255, np.uint8)])
    cv2.rectangle(card, (0, 0), (px - 1, px - 1), (255, 255, 255, 255), 10)
    return prepare(card)


def draw_slot(frame, raw, tr, kp, now):
    s = tr.slot
    if s is None:
        return
    el = now - s["t0"]
    k = int(np.searchsorted(s["times"], el, side="right"))
    done = k >= len(s["seq"])
    item = s["target"] if done else s["seq"][k]

    # Animación: algo más pequeña mientras gira, con "rebote" al revelar el resultado
    pop = el - s["times"][-1]
    scale = 1.0 + 0.25 * np.exp(-6 * pop) * np.cos(14 * pop) if done else 0.9

    mid, up, angle, iod, _ = face_pose(kp)
    if item == MONKEY:
        sprite, w = make_card(monkey_img(raw, kp)), CARD_PX
    else:
        sprite, w = PROPS[item]
    anchor = np.array([sprite.shape[1] / 2, sprite.shape[0] / 2])
    sc = CARD_W * iod / w * scale
    dst = mid + up * LIFT * iod
    overlay(frame, sprite, anchor, dst, sc, sc, angle)

    if TITLE is not None:
        th, tw = TITLE.shape[:2]
        st = CARD_W * 1.1 * iod / tw
        t_dst = mid + up * (LIFT + CARD_W / 2 + 0.45) * iod
        overlay(frame, TITLE, np.array([tw / 2, th / 2]), t_dst, st, st, angle)

Bucle de ejecución principal

In [10]:
face = vision.FaceLandmarker.create_from_options(vision.FaceLandmarkerOptions(
    base_options=mp_python.BaseOptions(model_asset_path=FACE_MODEL),
    running_mode=vision.RunningMode.VIDEO, num_faces=MAX_FACES))

# Índices de la malla facial de Mediapipe
EYE_A, EYE_B = (33, 133), (362, 263)
NOSE_I, MOUTH_A, MOUTH_B = 1, 61, 291
_last_ts = -1


def detect(frame, t):
    """Lista de dicts con los 5 landmarks de cada cara, sacados de la malla facial de Mediapipe."""
    global _last_ts
    ts = max(int(t * 1000), _last_ts + 1)
    _last_ts = ts
    res = face.detect_for_video(
        mp.Image(image_format=mp.ImageFormat.SRGB, data=cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)), ts)
    H, W = frame.shape[:2]
    out = []
    for lm in res.face_landmarks:
        p = lambda i: np.array([lm[i].x * W, lm[i].y * H])
        out.append({"left_eye": (p(EYE_A[0]) + p(EYE_A[1])) / 2,
                    "right_eye": (p(EYE_B[0]) + p(EYE_B[1])) / 2,
                    "nose": p(NOSE_I), "mouth_left": p(MOUTH_A), "mouth_right": p(MOUTH_B)})
    return out


cap = cv2.VideoCapture(0)
cap.set(3, 640)
cap.set(4, 480)

tracks = []

while True:
    ret, frame = cap.read()
    if not ret:
        break
    frame = cv2.flip(frame, 1)
    raw = frame.copy()
    t = time.perf_counter()

    faces = sorted(detect(frame, t), key=lambda kp: -np.linalg.norm(kp["left_eye"] - kp["right_eye"]))[:MAX_FACES]
    update_tracks(tracks, faces, t)

    for tr in tracks:
        kp = tr.kp(t)
        if AUTO_START and tr.slot is None:
            start_spin(tr, t)
        draw_slot(frame, raw, tr, kp, t)

    cv2.imshow("Filtro ruleta", frame)

    tecla = cv2.waitKey(1) & 0xFF
    if tecla in (ord("q"), 27):
        break
    elif tecla == ord(" "):
        for tr in tracks:
            start_spin(tr, t)

face.close()
cap.release()
if _mcap is not None:
    _mcap.release()
cv2.destroyAllWindows()